<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ Qwen3-TTS — Voice Clone, Custom & Design</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Advanced Text-to-Speech AI — Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 20px 0;'>Free on Google Colab T4 GPU | Open Source | 1.7B Model</p>
</div>

---

<div align="center">


  <img src="https://img.shields.io/badge/Colab-Free%20Tier-orange?style=for-the-badge&logo=googlecolab&logoColor=white" />
  <img src="https://img.shields.io/badge/Model-Qwen3--TTS%201.7B-blue?style=for-the-badge" />
<a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

**Notebook by [AIQUEST Academy](https://youtube.com/@AIQuestAcademy) — Subscribe for more AI tools & tutorials!**

| Feature | Description |
|---------|-------------|
| 🎤 Voice Cloning | Clone any voice with 3+ seconds of audio |
| 🎭 Custom Voice | 9 preset character voices across 9 languages |
| 🎨 Voice Design | Design unique voices from text descriptions |
| ⚡ Optimized | bfloat16 + SDPA attention — no flash-attn needed |

In [ ]:
# @title Step 1: 🔍 GPU Check & PyTorch Verification
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    try:
        x = torch.zeros(1, device="cuda")
        del x
        print("\n✅ PyTorch CUDA is working!")
    except Exception as e:
        print(f"\n⚠️ CUDA test failed: {e}")
else:
    print("\n❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title Step 2: 📦 Install Dependencies
print("📦 Installing Qwen3-TTS dependencies...")

# qwen-tts 0.1.1 requires transformers==4.57.3 exactly
!pip install -q "transformers==4.57.3" "accelerate>=1.2.0" "soundfile" "gradio>=5.0.0" "huggingface_hub"
!pip install -q qwen-tts

print("\n✅ All dependencies installed!")
print("─" * 50)

import torch
print(f"🔧 torch: {torch.__version__}")
print(f"🔧 CUDA available: {torch.cuda.is_available()}")
try:
    import transformers; print(f"🔧 transformers: {transformers.__version__}")
except: print("⚠️ transformers: NOT FOUND")
try:
    from qwen_tts import Qwen3TTSModel; print("🔧 qwen_tts: ✅ imported successfully")
except Exception as e: print(f"⚠️ qwen_tts import failed: {e}")

In [ ]:
# @title Step 3: 🚀 Load Model & Launch Gradio UI

import gradio as gr
from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import tempfile
import gc
import time

# ── PyTorch Optimizations ──
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ── Global State ──
current_model = None
current_model_type = None

MODEL_MAP = {
    "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
}


def load_model(model_type):
    """Load model with SDPA optimization, swap if different type needed"""
    global current_model, current_model_type

    if current_model_type == model_type:
        print(f"✅ Using cached {model_type} model")
        return current_model

    if current_model is not None:
        print(f"♻️ Unloading {current_model_type} model...")
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    model_name = MODEL_MAP[model_type]
    print(f"📥 Loading {model_name}...")
    start = time.time()

    try:
        current_model = Qwen3TTSModel.from_pretrained(
            model_name,
            device_map="cuda:0",
            dtype=torch.float16,
            attn_implementation="sdpa",
        )

        current_model_type = model_type
        load_time = time.time() - start
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        print(f"✅ Loaded in {load_time:.1f}s | GPU: {allocated:.2f} GB")
        return current_model

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        import traceback; traceback.print_exc()
        return None


def voice_clone(text, reference_audio, ref_transcript, use_fast_mode):
    """Generate speech by cloning a reference voice"""
    if not text or not reference_audio:
        return None

    try:
        total_start = time.time()
        model = load_model("base")
        if model is None:
            return None

        print("⏱️ Creating prompt...")
        prompt_start = time.time()

        if use_fast_mode or not ref_transcript:
            prompt_items = model.create_voice_clone_prompt(
                ref_audio=reference_audio,
                x_vector_only_mode=True
            )
        else:
            prompt_items = model.create_voice_clone_prompt(
                ref_audio=reference_audio,
                ref_text=ref_transcript,
                x_vector_only_mode=False
            )

        prompt_time = time.time() - prompt_start
        print(f"   Prompt: {prompt_time:.1f}s")

        print("⏱️ Generating audio...")
        gen_start = time.time()

        with torch.inference_mode():
            wavs, sr = model.generate_voice_clone(
                text=text,
                voice_clone_prompt=prompt_items
            )

        gen_time = time.time() - gen_start

        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(temp_file.name, wavs[0], sr)

        total_time = time.time() - total_start
        audio_duration = len(wavs[0]) / sr
        rtf = gen_time / audio_duration if audio_duration > 0 else 0

        print(f"✅ Done! Total: {total_time:.1f}s | Gen: {gen_time:.1f}s | Audio: {audio_duration:.1f}s | RTF: {rtf:.2f}x")

        torch.cuda.empty_cache()
        gc.collect()
        return temp_file.name

    except Exception as e:
        print(f"❌ Error in voice_clone: {e}")
        import traceback; traceback.print_exc()
        return None


def custom_voice(text, voice_name, instruction):
    """Generate speech using preset voices"""
    if not text:
        return None

    try:
        total_start = time.time()
        model = load_model("custom")
        if model is None:
            return None

        print(f"⏱️ Generating with voice: {voice_name}...")

        gen_start = time.time()

        with torch.inference_mode():
            if instruction and instruction.strip():
                print(f"   Style: '{instruction}'")
                wavs, sr = model.generate_custom_voice(
                    text=text,
                    speaker=voice_name,
                    instruct=instruction
                )
            else:
                wavs, sr = model.generate_custom_voice(
                    text=text,
                    speaker=voice_name
                )

        gen_time = time.time() - gen_start

        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(temp_file.name, wavs[0], sr)

        total_time = time.time() - total_start
        audio_duration = len(wavs[0]) / sr
        rtf = gen_time / audio_duration if audio_duration > 0 else 0

        print(f"✅ Done! Total: {total_time:.1f}s | Gen: {gen_time:.1f}s | Audio: {audio_duration:.1f}s | RTF: {rtf:.2f}x")

        torch.cuda.empty_cache()
        gc.collect()
        return temp_file.name

    except Exception as e:
        print(f"❌ Error in custom_voice: {e}")
        import traceback; traceback.print_exc()
        return None


def voice_design(text, voice_description):
    """Generate speech from text description of desired voice"""
    if not text or not voice_description:
        return None

    try:
        total_start = time.time()
        model = load_model("design")
        if model is None:
            return None

        print("⏱️ Generating with voice design...")
        gen_start = time.time()

        with torch.inference_mode():
            wavs, sr = model.generate_voice_design(
                text=text,
                instruct=voice_description
            )

        gen_time = time.time() - gen_start

        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(temp_file.name, wavs[0], sr)

        total_time = time.time() - total_start
        audio_duration = len(wavs[0]) / sr
        rtf = gen_time / audio_duration if audio_duration > 0 else 0

        print(f"✅ Done! Total: {total_time:.1f}s | Gen: {gen_time:.1f}s | Audio: {audio_duration:.1f}s | RTF: {rtf:.2f}x")

        torch.cuda.empty_cache()
        gc.collect()
        return temp_file.name

    except Exception as e:
        print(f"❌ Error in voice_design: {e}")
        import traceback; traceback.print_exc()
        return None


# ══════════════════════════════════════════════
# Gradio Interface
# ══════════════════════════════════════════════

with gr.Blocks(
    title="Qwen3-TTS - By AIQuest Academy",
    theme=gr.themes.Soft(),
) as demo:

    # Branding Header
    gr.HTML("""
        <div style="text-align:center; padding:25px;
            background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);
            border-radius:12px; box-shadow:0 8px 16px rgba(0,0,0,0.2); margin-bottom:20px;">
            <h1 style="font-size:2.2em; margin:0; color:white;">🎙️ Qwen3-TTS 1.7B</h1>
            <p style="font-size:1.1em; color:rgba(255,255,255,0.9); margin:6px 0 12px;">
                Voice Cloning | Custom Voices | Voice Design</p>
            <div style="display:flex; justify-content:center; gap:12px; flex-wrap:wrap;">
                <a href="https://www.youtube.com/@AiQuestacademy?sub_confirmation=1" target="_blank"
                   style="background:#FF0000; color:#fff; padding:8px 18px; border-radius:8px;
                   text-decoration:none; font-weight:600; font-size:0.95em;">
                   ▶ YouTube - AIQUEST</a>
                <a href="https://x.com/AiQuestacademy" target="_blank"
                   style="background:#000; color:#fff; padding:8px 18px; border-radius:8px;
                   text-decoration:none; font-weight:600; font-size:0.95em;">
                   𝕏 Follow on X</a>
            </div>
            <p style="margin-top:10px; font-size:0.85em; color:rgba(255,255,255,0.7);">
                Notebook by <b>AIQUEST</b> — Subscribe for more AI tools &amp; tutorials!</p>
        </div>
    """)

    with gr.Tab("🎤 Voice Cloning"):
        gr.Markdown("### Clone any voice with 3+ seconds of audio")
        with gr.Row():
            with gr.Column():
                clone_text = gr.Textbox(label="Text to Synthesize", placeholder="Enter text (shorter = faster)...", lines=4)
                clone_audio = gr.Audio(label="Reference Audio (3+ seconds)", type="filepath")
                clone_transcript = gr.Textbox(label="Transcript (Optional - improves quality)", placeholder="What's said in the audio...", lines=2)
                clone_fast_mode = gr.Checkbox(label="Fast Mode (skip transcript)", value=True)
                clone_btn = gr.Button("🎵 Generate Speech", variant="primary", size="lg")
            with gr.Column():
                clone_output = gr.Audio(label="Generated Speech")
                gr.Markdown("**First run: ~2-3 min (model load) | RTF: 3.5-5x**")

        clone_btn.click(voice_clone, inputs=[clone_text, clone_audio, clone_transcript, clone_fast_mode], outputs=clone_output)

    with gr.Tab("🎭 Custom Voice"):
        gr.Markdown("### 9 preset character voices with style control")
        with gr.Row():
            with gr.Column():
                custom_text = gr.Textbox(label="Text to Synthesize", placeholder="Enter text...", lines=4)
                custom_voice_name = gr.Dropdown(
                    choices=["Serena", "Vivian", "Ono_Anna", "Sohee", "Aiden", "Dylan", "Eric", "Ryan", "Uncle_Fu"],
                    label="Voice Character", value="Serena"
                )
                custom_instruction = gr.Textbox(label="Style Instruction (Optional)", placeholder="e.g., 'speak slowly and cheerfully'", lines=2)
                gr.Markdown("**Female**: Serena, Vivian, Ono_Anna, Sohee · **Male**: Aiden, Dylan, Eric, Ryan, Uncle_Fu")
                custom_btn = gr.Button("🎵 Generate Speech", variant="primary", size="lg")
            with gr.Column():
                custom_output = gr.Audio(label="Generated Speech")
                gr.Markdown("**RTF: 3.5-5x**")

        custom_btn.click(custom_voice, inputs=[custom_text, custom_voice_name, custom_instruction], outputs=custom_output)

    with gr.Tab("🎨 Voice Design"):
        gr.Markdown("### Design a unique voice from text description")
        with gr.Row():
            with gr.Column():
                design_text = gr.Textbox(label="Text to Synthesize", placeholder="Enter text...", lines=4)
                design_description = gr.Textbox(label="Voice Description", placeholder="A young female, cheerful, speaking clearly with British accent", lines=3)
                gr.Markdown("**Tips**: Describe age, gender, emotion, accent, speed")
                design_btn = gr.Button("🎵 Generate Speech", variant="primary", size="lg")
            with gr.Column():
                design_output = gr.Audio(label="Generated Speech")
                gr.Markdown("**RTF: 3.5-5x**")

        design_btn.click(voice_design, inputs=[design_text, design_description], outputs=design_output)

    # Footer
    gr.HTML("""
        <div style="text-align:center; padding:15px; background:linear-gradient(135deg,#f5f7fa 0%,#c3cfe2 100%);
            border-radius:8px; margin-top:20px; font-size:0.9em; color:#555;">
            <p><strong style="color:#667eea;">🎓 Created by AIQuest Academy</strong></p>
            <p>📺 Subscribe for more AI tutorials, tools & cutting-edge tech!</p>
            <p style="margin-top:8px;">
                <a href="https://www.youtube.com/@AIQuestAcademy" target="_blank" style="color:#667eea; text-decoration:none; font-weight:bold;">YouTube: @AIQuestAcademy</a> |
                <a href="https://twitter.com/AIQuestAcademy" target="_blank" style="color:#667eea; text-decoration:none; font-weight:bold;">X: @AIQuestAcademy</a>
            </p>
            <p style="margin-top:10px; font-size:0.85em; color:#888;">Powered by Qwen3-TTS 1.7B | Google Colab | Free & Open Source</p>
        </div>
    """)

print("=" * 60)
print("🎙️ Qwen3-TTS 1.7B Notebook")
print("📺 Created by: AIQuest Academy")
print("🔗 YouTube: @AIQuestAcademy | X: @AIQuestAcademy")
print("=" * 60)
print("\n🚀 Launching Gradio UI...")
demo.launch(share=True, debug=True)

---

<div align="center">

### 🎉 Enjoyed this notebook?

If this was helpful, please **⭐ star the repo** and **subscribe** for more free AI tools & tutorials!

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

**Made with ❤️ by AIQUEST** | [@aiquestacademy](https://youtube.com/@aiquestacademy)

</div>